# Future workflow: sort + manual GUI curation in **one** environment (Spyglass v2 preview)

This notebook is a **preview of the workflow that becomes possible once Spyglass
[PR #1609](https://github.com/LorenFrankLab/spyglass/pull/1609) is merged**. It runs the lab's
favorite workflow -- automatic spike sorting followed by **manual curation in the SpikeInterface
desktop GUI** -- end to end, in a **single conda environment and a single notebook**.

## Environment requirements

Run this in the **`spyglass` conda environment** with the **PR #1609 branch** (`spikesorting-v2`)
installed -- that branch provides SpikeInterface 0.104 and the `spyglass.spikesorting.v2` pipeline.
Two extra packages must be installed into that environment:

```bash
pip install mountainsort5            # the sorter
pip install "spikeinterface-gui[desktop]"   # the desktop curation GUI
```

The whole point of this notebook is that **`spikeinterface-gui` now coexists with Spyglass** in one
environment (it is compatible with SpikeInterface 0.104). Once PR #1609 ships, making this a
documented optional dependency would remove even this manual install.

## Connect to the database

We load the DataJoint config and import the **v2** spike-sorting module. Importing
`spyglass.common` opens the connection.

In [1]:
import json

import datajoint as dj

# Load config for database connection info
dj_local_conf_path = "/Users/pauladkisson/Documents/CatalystNeuro/Spyglass/spyglass/dj_local_conf.json"
dj.config.load(dj_local_conf_path)

# Spyglass stores some parameters as native Python objects (dicts / lists) in the database;
# this flag lets DataJoint serialize and deserialize those blobs instead of rejecting them.
dj.config["enable_python_native_blobs"] = True

# General Spyglass imports (importing common connects to the database)
import spyglass.common as sgc
import spyglass.spikesorting.v2 as sgs2
from spyglass.spikesorting.v2 import initialize_v2_defaults
from spyglass.spikesorting.v2.recording import SortGroupV2
from spyglass.spikesorting.v2.sorting import Sorting
from spyglass.spikesorting.v2.curation import CurationV2
from spyglass.spikesorting.v2.pipeline import (
    preflight_v2_pipeline,
    run_v2_pipeline,
    describe_pipeline_presets,
    describe_sort_groups,
)
from spyglass.spikesorting.v2._analyzer_cache import analyzer_path
from spyglass.spikesorting.spikesorting_merge import SpikeSortingOutput
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename

/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/datajoint/plugin.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources  # requires setuptools<82
[2026-06-18 06:54:24,127][INFO]: DataJoint is configured from /Users/pauladkisson/Documents/CatalystNeuro/Spyglass/spyglass/dj_local_conf.json
[2026-06-18 06:54:24,523][INFO]: DataJoint 0.14.9 connected to root@localhost:3306
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/sorters/external/kilosort4.py:110: UserWarning: Kilosort4 is not installed. Please install kilosort4 to get the parameters.
  warnings.warn("Kilosort4 is not installed. Please install kilosort4 to get the parameters.")


## Parameters set manually

Everything you might change for a different session, shank, or sorter lives here. The v2 pipeline
bundles the per-stage parameter choices into a single named **pipeline preset**; run
`describe_pipeline_presets()` to see what each one does.

In [2]:
### Parameters set manually ###

# Session: the NWB copy that lives in the database (note the trailing underscore)
nwb_file_name = get_nwb_copy_filename("H3022-210806.nwb")  # -> "H3022-210806_.nwb"

# Which shank (sort group) and which epoch (interval) to sort
sort_group_id = 0
interval_list_name = "01"  # first wake epoch for this session

# Lab team that owns the sorting (any unique name)
team_name = "Wood/Dudchenko Lab"

# v2 pipeline preset: a named bundle of preprocessing + artifact + sorter parameter rows.
# 'franklab_tetrode_mountainsort5' uses mountainsort5 (requires: pip install mountainsort5).
pipeline_preset = "franklab_tetrode_mountainsort5"

describe_pipeline_presets()

,pipeline_preset,sorter,preprocessing_params_name,artifact_detection_params_name,sorter_params_name,intended_use,threshold_units,notes
0,franklab_tetrode_clusterless_thresholder,clusterless_thresholder,default_franklab,default,default,Peak detection only (no clustering); feeds the...,µV (100 µV default),The 'default' clusterless SorterParameters row...
1,franklab_tetrode_mountainsort4,mountainsort4,default_franklab,default,franklab_tetrode_hippocampus_30kHz_ms4,Frank-lab hippocampal tetrodes at 30 kHz; lega...,MAD multiplier,MountainSort detect_threshold is a median-abso...
2,franklab_tetrode_mountainsort5,mountainsort5,default_franklab,default,franklab_tetrode_hippocampus_30kHz_ms5,Frank-lab hippocampal tetrodes at 30 kHz; reco...,MAD multiplier,MountainSort detect_threshold is a median-abso...


## Confirm the session and lab team

A quick check that the session is ingested, and creation of the owning `LabTeam` if needed.

In [3]:
if not (sgc.LabTeam() & {"team_name": team_name}):
    sgc.LabTeam().create_new_team(
        team_name=team_name,
        team_members=[],
        team_description="Wood/Dudchenko lab spike sorting",
    )

sgc.Session() & {"nwb_file_name": nwb_file_name}

nwb_file_name name of the NWB file,subject_id,institution_name,lab_name,session_id,session_description,session_start_time,timestamps_reference_time,experiment_description
H3022-210806_.nwb,H3022,University of Edinburgh,Wood/Dudchenko lab,210806,"Exploration, sleep and cue rotation",2021-08-06 11:34:07,2021-08-06 11:34:07,Basic properties of the head-direction system


## Install the v2 default parameters and define sort groups

`initialize_v2_defaults()` seeds every default parameter row the pipeline needs in one call.

Sort groups bundle the electrodes that are sorted together (one per shank here). The v2
`SortGroupV2.set_group_by_shank` is **tolerant of non-numeric electrode-group names** like
`probe1_shank1` -- a real improvement over the v1 helper, which assumed numeric names and so
required building sort groups by hand for this dataset. We only create the groups if none exist
yet (v2 refuses to silently overwrite existing sort groups on rerun).

In [4]:
initialize_v2_defaults()

if not (SortGroupV2 & {"nwb_file_name": nwb_file_name}):
    SortGroupV2.set_group_by_shank(nwb_file_name=nwb_file_name)

describe_sort_groups(nwb_file_name)

[06:54:33][INFO] Spyglass: SorterParameters.insert_default: skipping default row 'franklab_tetrode_hippocampus_30kHz_ms4' -- sorter 'mountainsort4' is not in spikeinterface.sorters.installed_sorters() on this platform.
[06:54:33][INFO] Spyglass: SorterParameters.insert_default: skipping default row 'franklab_probe_ctx_30kHz_ms4' -- sorter 'mountainsort4' is not in spikeinterface.sorters.installed_sorters() on this platform.
[06:54:33][INFO] Spyglass: SorterParameters.insert_default: skipping default row 'franklab_tetrode_hippocampus_30KHz' -- sorter 'mountainsort4' is not in spikeinterface.sorters.installed_sorters() on this platform.
[06:54:33][INFO] Spyglass: SorterParameters.insert_default: skipping default row 'franklab_probe_ctx_30KHz' -- sorter 'mountainsort4' is not in spikeinterface.sorters.installed_sorters() on this platform.
[06:54:33][INFO] Spyglass: SorterParameters.insert_default: skipping default row 'franklab_neuropixels_default' -- sorter 'kilosort4' is not in spikeint

,nwb_file_name,sort_group_id,n_electrodes,electrode_ids,electrode_group_names,probe_shanks,brain_regions,bad_channel_count,reference_mode,reference_electrode_id
0,H3022-210806_.nwb,0,32,"(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","(probe1_shank1,)","(1,)","(left postsubiculum,)",0,none,None


## Optional: reset to re-run the sort

`run_v2_pipeline` is idempotent -- re-running with the same inputs reuses the existing rows. Run
this cell only if you want a clean re-sort. It uses `preflight_v2_pipeline` to resolve the
deterministic `recording_id` / `sorting_id` this configuration *would* produce, then deletes just
those rows (cascading through `Recording` → `Sorting` → `CurationV2`) and sweeps the now-orphaned
`SpikeSortingOutput` master rows. Everything else is left untouched. **Skip on a normal first run.**

In [7]:
# OPTIONAL RESET -- run only if you want to re-sort from scratch.
from spyglass.spikesorting.v2.recording import RecordingSelection

report = preflight_v2_pipeline(
    nwb_file_name=nwb_file_name,
    sort_group_id=sort_group_id,
    interval_list_name=interval_list_name,
    team_name=team_name,
    pipeline_preset=pipeline_preset,
)
recording_id = report.expected_ids["recording_id"]["id"]
sorting_id = report.expected_ids["sorting_id"]["id"]

reset_recording = RecordingSelection & {"recording_id": recording_id}
if reset_recording:
    # Capture the merge masters before the cascade removes their part rows.
    reset_merge_ids = (
        SpikeSortingOutput.CurationV2 & {"sorting_id": sorting_id}
    ).fetch("merge_id")

    # Cascade-delete recording -> sorting -> curation (also drops the CurationV2 part rows).
    reset_recording.delete()

    # Sweep the now-orphaned SpikeSortingOutput master rows.
    if len(reset_merge_ids):
        (SpikeSortingOutput & [{"merge_id": m} for m in reset_merge_ids]).super_delete(
            warn=False, safemode=False
        )
else:
    print("Nothing to reset for this configuration.")

[2026-06-17 16:20:30,767][INFO]: Deleting 1 rows from `spikesorting_merge`.`spike_sorting_output__curation_v2`
[2026-06-17 16:20:30,771][INFO]: Deleting 1 rows from `spikesorting_merge`.`spike_sorting_output`
[2026-06-17 16:20:30,794][INFO]: Deleting 28 rows from `spikesorting_v2_curation`.`curation_v2__merge_group`
[2026-06-17 16:20:30,799][INFO]: Deleting 28 rows from `spikesorting_v2_curation`.`curation_v2__unit`
[2026-06-17 16:20:30,804][INFO]: Deleting 1 rows from `spikesorting_v2_curation`.`curation_v2`
[2026-06-17 16:20:30,809][INFO]: Deleting 0 rows from `spikesorting_v2_curation`.`curation_v2__unit`
[2026-06-17 16:20:30,815][INFO]: Deleting 0 rows from `spikesorting_v2_curation`.`curation_v2`
[2026-06-17 16:20:30,823][INFO]: Deleting 28 rows from `spikesorting_v2_sorting`.`__sorting__unit`
[2026-06-17 16:20:30,828][INFO]: Deleting 1 rows from `spikesorting_v2_sorting`.`__sorting`
[2026-06-17 16:20:30,836][INFO]: Deleting 1 rows from `spikesorting_v2_sorting`.`sorting_selection

## Run the v2 pipeline

`preflight_v2_pipeline` is a ~1-second, read-only check that every prerequisite (session, interval,
team, sort group, parameter rows, sorter binary) is in place -- it fails fast with the exact fix
instead of erroring minutes into `populate`. Then `run_v2_pipeline` chains
recording → artifact detection → sort → root curation into one call and returns a manifest of the
ids it produced (including the `merge_id` downstream pipelines key off).

> ⏱️ **Long-running cell (~30 minutes).** Filtering / referencing the recording dominates; the
> sort itself takes about a minute.

In [5]:
report = preflight_v2_pipeline(
    nwb_file_name=nwb_file_name,
    sort_group_id=sort_group_id,
    interval_list_name=interval_list_name,
    team_name=team_name,
    pipeline_preset=pipeline_preset,
)
assert report.ok, f"Preflight failed:\n" + "\n".join(report.errors)

manifest = run_v2_pipeline(
    nwb_file_name=nwb_file_name,
    sort_group_id=sort_group_id,
    interval_list_name=interval_list_name,
    team_name=team_name,
    pipeline_preset=pipeline_preset,
    description="future-workflow single-environment demo",
)

sorting_id = manifest["sorting_id"]
root_curation_id = manifest["curation_id"]
print(f"sorting_id      = {sorting_id}")
print(f"root curation   = {root_curation_id}")
print(f"merge_id        = {manifest['merge_id']}")
print(f"n_units         = {manifest['n_units']}")

[06:54:52][WARNING] Spyglass: CurationV2.insert_curation: root curation already exists for sorting_id=294bd737-f773-56c4-ba49-bdd20bcd2556; returning existing key without staging a new NWB.


sorting_id      = 294bd737-f773-56c4-ba49-bdd20bcd2556
root curation   = 0
merge_id        = be6e8af1-f41f-41d0-4a06-28095922898d
n_units         = 28


## Build the SortingAnalyzer (in this environment)

This is the step that previously forced a kernel switch. `Sorting.get_analyzer` returns a
SpikeInterface `SortingAnalyzer` built directly from the database row -- **no export, no second
environment**.

The analyzer the pipeline builds carries a core extension set (`random_spikes`, `noise_levels`,
`templates`, `waveforms`). The SpikeInterface GUI is more useful with a few more extensions, so we
compute them here.

> ⏱️ A few minutes to compute the extra extensions the first time.

In [6]:
analyzer = Sorting().get_analyzer({"sorting_id": sorting_id})

# Extensions the GUI uses beyond the core set the pipeline already computed.
analyzer.compute(
    ["spike_amplitudes", "correlograms", "unit_locations", "template_similarity"]
)
analyzer.compute("quality_metrics")

analyzer_folder = analyzer_path(sorting_id)
print(f"analyzer folder: {analyzer_folder}")
analyzer

/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/sortinganalyzer.py:2504: UserWarning: Found no run_info file for spike_amplitudes, extension should be re-computed.
  warnings.warn(f"Found no run_info file for {self.extension_name}, extension should be re-computed.")
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/sortinganalyzer.py:2586: UserWarning: Found no data for spike_amplitudes, extension should be re-computed.
  warnings.warn(f"Found no data for {self.extension_name}, extension should be re-computed.")
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/postprocessing/template_similarity.py:345: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  overlapping_ids = overlapping_j_list[i]


Compute : spike_amplitudes (no parallelization):   0%|          | 0/1220 [00:00<?, ?it/s]

/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/core/analyzer_extension_core.py:1165: UserWarning: The following metrics will not be computed due to missing dependencies: ['nearest_neighbor', 'mahalanobis', 'drift', 'd_prime', 'silhouette']
  warnings.warn(
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/spikeinterface/metrics/quality/misc_metrics.py:1026: UserWarning: Amplitude cutoff set to NaN fo

analyzer folder: /Users/pauladkisson/Documents/CatalystNeuro/DudchenkoConv/Spyglass/tmp/spikesorting_v2/analyzers/294bd737-f773-56c4-ba49-bdd20bcd2556.analyzer


SortingAnalyzer: 32 channels - 28 units - 1 segments - binary_folder - sparse - has recording
Loaded 9 extensions: random_spikes, unit_locations, waveforms, spike_amplitudes, templates, correlograms, template_similarity, noise_levels, quality_metrics

## Manual curation in the SpikeInterface GUI

Launch the desktop GUI on the analyzer we just built. In the GUI you can **merge** over-split
units, **label** units (`good` / `MUA` / `noise`), and **remove** noise units. When you are done,
use the GUI's **save in analyzer** action -- it writes the curation to:

```
<analyzer_folder>/spikeinterface_gui/curation_data.json
```

This cell **blocks** until you close the GUI window, so run it interactively (it needs a display).
If you prefer to launch the GUI from a terminal instead, the equivalent command is:

```bash
sigui <analyzer_folder> --curation --mode desktop
```

In [7]:
from spikeinterface_gui import run_mainwindow

run_mainwindow(analyzer, mode="desktop", curation=True)

<spikeinterface_gui.backend_qt.QtMainWindow(0x3a1eb19b0) at 0x3977e0a00>

## Re-ingest the GUI curation into Spyglass

We read the GUI's `curation_data.json` and translate it into a `CurationV2` row that branches off
the root curation produced by the pipeline. The GUI's label vocabulary (`good` / `MUA` / `noise`)
is mapped to Spyglass's `CurationLabel` set (`accept` / `mua` / `noise` / `artifact` / `reject`),
removed units are labeled `reject`, and the GUI merge groups are passed straight through.

In [8]:
# GUI label -> Spyglass CurationLabel
LABEL_MAP = {"good": "accept", "noise": "noise", "MUA": "mua", "mua": "mua"}

curation_json_path = analyzer_folder / "spikeinterface_gui" / "curation_data.json"
with open(curation_json_path) as file:
    gui_curation = json.load(file)

# Per-unit labels from manual_labels (flatten the per-category label lists).
labels = {}
for entry in gui_curation.get("manual_labels", []):
    flat = [
        LABEL_MAP.get(label, label)
        for label_list in entry["labels"].values()
        for label in label_list
    ]
    if flat:
        labels[entry["unit_id"]] = flat

# Units removed in the GUI are recorded as 'reject'.
for unit_id in gui_curation.get("removed", []):
    labels[unit_id] = ["reject"]

# Merge groups: each GUI merge contributes a list of unit_ids to merge.
merge_groups = [merge["unit_ids"] for merge in gui_curation.get("merges", [])]

manual_curation_key = CurationV2.insert_curation(
    sorting_key={"sorting_id": sorting_id},
    labels=labels or None,
    merge_groups=merge_groups or None,
    apply_merge=False,  # record the merges (reviewable) without committing them
    parent_curation_id=root_curation_id,
    curation_source="manual",
    description="manual curation (SpikeInterface GUI)",
)
manual_curation_key

/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/backends/hdf5/h5tools.py:620: BrokenLinkWarning: Path to Group altered/broken at /general/devices/camera_device 0/model
  warnings.warn('Path to Group altered/broken at ' + os.path.join(h5obj.name, k), BrokenLinkWarning)
[07:04:50][INFO] Spyglass: Writing new NWB file H3022-210806_JFMSNKKPKT.nwb
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/build/objectmapper.py:738: MissingRequiredBuildWarning: NWBFile 'root' is missing required value for attribute 'source_script_file_name'.
  warnings.warn(msg, MissingRequiredBuildWarning)
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/backends/hdf5/h5tools.py:620: BrokenLinkWarning: Path to Group altered/broken at /general/devices/camera_device 0/model
  warnings.warn('Path to Group altered/broken at ' + os.path.join(h5obj.name, k), BrokenLinkWarning)
/opt/anaconda3/envs/spyglass/lib/python3.10/site-packages/hdmf/build/objectmapper.py:738: MissingRequired

{'sorting_id': UUID('294bd737-f773-56c4-ba49-bdd20bcd2556'), 'curation_id': 1}

## Summarize and expose the curated result

`summarize_curation` returns a notebook-printable summary of the curation (unit counts, labels,
merge groups, and the `merge_id` it was registered under). The same curation now appears on the
`SpikeSortingOutput` merge table, ready for any downstream pipeline.

In [9]:
from pprint import pprint

pprint(CurationV2.summarize_curation(manual_curation_key))

SpikeSortingOutput.CurationV2 & manual_curation_key

{'curation_id': 1,
 'description': 'manual curation (SpikeInterface GUI)',
 'is_merge_preview': False,
 'labels': {1: ['reject']},
 'merge_groups': {1: [1],
                  2: [2],
                  4: [4],
                  5: [5],
                  6: [6],
                  7: [7],
                  8: [8],
                  9: [9],
                  10: [10],
                  11: [11],
                  13: [13],
                  14: [14],
                  15: [15],
                  16: [16],
                  17: [17],
                  18: [18],
                  19: [19],
                  20: [20],
                  21: [21],
                  22: [22],
                  23: [23],
                  24: [24],
                  25: [25],
                  26: [26],
                  27: [27],
                  28: [28],
                  29: [29],
                  30: [30]},
 'merge_id': UUID('c4ebe340-3781-7896-2b87-2d2d21eecb96'),
 'merges_applied': False,
 'n_units': 28,

merge_id,sorting_id,curation_id
c4ebe340-3781-7896-2b87-2d2d21eecb96,294bd737-f773-56c4-ba49-bdd20bcd2556,1
